# Finetuning de Sentiment Analysis: Normal vs LoRA

Este notebook entrena un modelo de *Transformers* sobre `data/sentiment_dataset.csv` con dos enfoques:

1. **Finetuning normal** (actualiza todos los parámetros)
2. **Finetuning con LoRA** (actualiza solo adaptadores de bajo rango)

También compara métricas de pérdida, precisión, tiempo y memoria, y deja ambos modelos listos para inferencia.


In [ ]:
# Instalación de dependencias clave para este notebook
!pip install -q transformers peft torch datasets evaluate accelerate scikit-learn pandas psutil


In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import torch
import psutil
import evaluate

from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

from peft import LoraConfig, TaskType, get_peft_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device detectado: {device}")


## 1) Carga y preparación del dataset


In [ ]:
candidate_paths = ["data/sentiment_dataset.csv", "../data/sentiment_dataset.csv"]
dataset_path = next((path for path in candidate_paths if os.path.exists(path)), None)
if dataset_path is None:
    raise FileNotFoundError(f"No se encontró el dataset en ninguna ruta: {candidate_paths}")

df = pd.read_csv(dataset_path)

# Limpieza mínima para asegurar calidad de texto/labels
df = df.dropna(subset=["text", "sentiment"]).copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""].reset_index(drop=True)

label_list = sorted(df["sentiment"].unique().tolist())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
df["label"] = df["sentiment"].map(label2id)

print("Shape:", df.shape)
print("Etiquetas:", label2id)
print(df["sentiment"].value_counts())


In [ ]:
# Split estratificado: train / validation / test (70/15/15)
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["label"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"],
)

dataset_dict = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df[["text", "label"]].reset_index(drop=True)),
        "validation": Dataset.from_pandas(val_df[["text", "label"]].reset_index(drop=True)),
        "test": Dataset.from_pandas(test_df[["text", "label"]].reset_index(drop=True)),
    }
)

for split_name, split_data in dataset_dict.items():
    print(f"{split_name}: {len(split_data)} muestras")


## 2) Tokenización


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 2


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)


tokenized_ds = dataset_dict.map(tokenize_function, batched=True)
tokenized_ds = tokenized_ds.remove_columns(["text"])
tokenized_ds.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenized_ds


## 3) Métricas y utilidades compartidas


In [ ]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    prec = precision_metric.compute(predictions=preds, references=labels, average="macro")["precision"]
    return {"accuracy": acc, "precision": prec}


def memory_usage_mb():
    cpu_mb = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
    gpu_mb = 0.0
    if torch.cuda.is_available():
        gpu_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    return cpu_mb, gpu_mb


def train_and_evaluate(model, output_dir):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_accuracy",
        greater_is_better=True,
        logging_steps=50,
        report_to="none",
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    start = time.time()
    train_result = trainer.train()
    train_time = time.time() - start

    val_metrics = trainer.evaluate(tokenized_ds["validation"])
    test_metrics = trainer.evaluate(tokenized_ds["test"])
    cpu_mb, gpu_mb = memory_usage_mb()

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    summary = {
        "train_loss": train_result.training_loss,
        "val_loss": val_metrics.get("eval_loss"),
        "val_accuracy": val_metrics.get("eval_accuracy"),
        "val_precision": val_metrics.get("eval_precision"),
        "test_loss": test_metrics.get("eval_loss"),
        "test_accuracy": test_metrics.get("eval_accuracy"),
        "test_precision": test_metrics.get("eval_precision"),
        "train_time_sec": train_time,
        "peak_cpu_mb": cpu_mb,
        "peak_gpu_mb": gpu_mb,
    }

    return trainer, summary


## 4) Finetuning Normal (todos los parámetros)


In [ ]:
full_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

full_output_dir = "../models/sentiment_full_finetuned"
full_trainer, full_metrics = train_and_evaluate(full_model, full_output_dir)

print("Métricas finetuning normal:")
for k, v in full_metrics.items():
    print(f"{k}: {v}")


## 5) Finetuning con LoRA (PEFT)


In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

lora_output_dir = "../models/sentiment_lora_finetuned"
lora_trainer, lora_metrics = train_and_evaluate(lora_model, lora_output_dir)

print("Métricas finetuning LoRA:")
for k, v in lora_metrics.items():
    print(f"{k}: {v}")


## 6) Comparativa de resultados


In [ ]:
comparison_df = pd.DataFrame(
    [
        {"approach": "full_finetuning", **full_metrics},
        {"approach": "lora_finetuning", **lora_metrics},
    ]
)

comparison_df[[
    "approach",
    "train_loss",
    "val_loss",
    "val_accuracy",
    "val_precision",
    "test_loss",
    "test_accuracy",
    "test_precision",
    "train_time_sec",
    "peak_cpu_mb",
    "peak_gpu_mb",
]]


## 7) Predicciones con ambos modelos


In [ ]:
def predict_sentiment(texts, trainer):
    encoded = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    model = trainer.model
    model.eval()
    model.to(device)
    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        logits = model(**encoded).logits
        probs = torch.softmax(logits, dim=-1)
        pred_ids = torch.argmax(probs, dim=-1).cpu().numpy()

    return [id2label[int(i)] for i in pred_ids], probs.cpu().numpy()


sample_texts = [
    "The company posted strong quarterly earnings and raised guidance.",
    "The stock fell after weaker than expected revenue.",
    "Markets remained flat as investors waited for the Fed decision.",
]

full_preds, _ = predict_sentiment(sample_texts, full_trainer)
lora_preds, _ = predict_sentiment(sample_texts, lora_trainer)

pd.DataFrame(
    {
        "text": sample_texts,
        "full_finetuning_pred": full_preds,
        "lora_finetuning_pred": lora_preds,
    }
)


## 8) Rutas de modelos guardados

- Modelo completo: `../models/sentiment_full_finetuned`
- Modelo LoRA (adaptadores + config): `../models/sentiment_lora_finetuned`

> Puedes reutilizar estas rutas para cargar los modelos en inferencia posterior.
